# The Physics of Play
## Aerodynamics in Summer Sports & FIFA World Cup 2026

Interactive computational fluid dynamics simulations demonstrating how aerodynamic forces shape athletic performance.

### Modules

1. **The World Cup Free-Kick (Magnus Effect)**: A rotating cylinder in crossflow generates an asymmetric pressure field and lateral force, modeling the curved trajectory of a spinning soccer ball.

2. **The Peloton & Running Pack (Wake Interaction)**: Velocity and pressure fields around single and multiple athlete silhouettes reveal the drag-reduction benefits of inline drafting and echelon formation.

Built with [ΦFlow](https://github.com/tum-pbs/PhiFlow) (v3.4+, JAX backend), a differentiable PDE solver on a 128×256 Cartesian grid.

## Governing Equations

### Incompressible Navier–Stokes

$$
\rho \left(\frac{\partial \mathbf{u}}{\partial t} + \mathbf{u} \cdot \nabla \mathbf{u}\right) = -\nabla p + \mu \nabla^2 \mathbf{u} + \mathbf{f}
$$

### Continuity (incompressibility constraint)

$$
\nabla \cdot \mathbf{u} = 0
$$

| Symbol | Quantity | Units |
|--------|----------|-------|
| $\mathbf{u}$ | Velocity field | m/s |
| $p$ | Pressure | Pa |
| $\rho$ | Density (air ≈ 1.225) | kg/m³ |
| $\mu$ | Dynamic viscosity (air ≈ 1.81×10⁻⁵) | Pa·s |
| $\mathbf{f}$ | Body forces (buoyancy, etc.) | N/m³ |

### Key dimensionless group

$$
Re = \frac{\rho U L}{\mu}
$$

For a soccer ball ($L \approx 0.22$ m) at $U \approx 30$ m/s:
$Re \approx 4.5 \times 10^5$ — well into the turbulent regime. Our 2D simulation at $Re \approx 10^3$ captures the essential wake physics at a feasible grid resolution.

In [ ]:
"""Imports and ΦFlow backend configuration."""
from phi.jax.flow import *
from phi import geom, math, field

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pathlib import Path
from tqdm import tqdm
from tqdm.notebook import trange
from IPython.display import HTML
import os, sys
from phi.physics import diffuse

# ── Dark theme ──
plt.style.use("dark_background")
matplotlib.rcParams.update(
    {
        "figure.facecolor": "#111111",
        "axes.facecolor": "#111111",
        "axes.edgecolor": "#cccccc",
        "axes.labelcolor": "#ffffff",
        "text.color": "#ffffff",
        "xtick.color": "#cccccc",
        "ytick.color": "#cccccc",
    }
)

# ── Output directory ──
ASSETS = Path("assets")
ASSETS.mkdir(exist_ok=True)

# Module 1: The World Cup Free-Kick

## Magnus Effect — Rotating Cylinder in Crossflow

A spinning soccer ball experiences a lateral force due to an asymmetric pressure field, the **Magnus effect**.

We model this by placing a rotating cylinder in a uniform crossflow. Clockwise rotation deflects the wake downward (for top-spin) or upward (for back-spin), bending the ball's trajectory mid-flight.

### Setup

- **Grid**: 256 × 128 cells spanning an 8 × 4 unit domain (wide rectangle, flow left→right)
- **Cylinder**: `geom.Sphere` radius 0.3, centered at (2.0, 2.0), angular velocity 10 rad/s
- **Inflow**: uniform 1.0 m/s from left boundary
- **Solver**: ΦFlow incompressible Navier-Stokes, 120 steps × Δt = 0.3

In [ ]:
# ── Module 1 — Run the Magnus effect simulation and collect data ─────────

# ── Domain & boundary conditions ──
BOX = geom.Box(x=8.0, y=4.0)
BOUNDARY = {
    "x-": vec(x=1.0, y=0.0),  # inflow
    "x+": ZERO_GRADIENT,  # outflow
    "y": 0,  # no-slip walls (top / bottom)
}

# ── Rotating cylinder obstacle ──
CYLINDER = geom.Sphere(x=2.0, y=2.0, radius=0.3)
CENTER = (2.0, 2.0)
RADIUS = 0.3
ANGULAR_VELOCITY = 10.0  # rad/s (positive = counter-clockwise → downward lift)

obstacle = Obstacle(
    CYLINDER, velocity=vec(x=0.0, y=0.0), angular_velocity=ANGULAR_VELOCITY
)

# ── Domain grid for visualisation ──
x_grid = np.linspace(0, 8.0, 256)
y_grid = np.linspace(0, 4.0, 128)

# ── Run simulation ──
v0 = StaggeredGrid((1.0, 0.0), BOUNDARY, x=256, y=128, bounds=BOX)
v, _ = fluid.make_incompressible(v0, ())

DT = 0.3
N_STEPS = 120
SAVE_EVERY = 4

frames = []
v = v0

for i in trange(N_STEPS):
    v = advect.semi_lagrangian(v, v, dt=DT)
    v, p = fluid.make_incompressible(v, obstacle)

    if i % SAVE_EVERY == 0:
        p_np = np.array(p.values.native("x", "y"))
        vel_mag = field.vec_length(v)
        vel_np = np.array(vel_mag.values.native("x", "y"))

        # Velocity components: staggered → centred
        vc = v @ CenteredGrid(0, x=256, y=128, bounds=BOX)
        u_c = np.array(vc.vector[0].values.native("x", "y"))
        v_c = np.array(vc.vector[1].values.native("x", "y"))

        frames.append(
            {
                "pressure": p_np,
                "velocity": vel_np,
                "u": u_c,
                "v": v_c,
                "step": i,
            }
        )

print(f"Captured {len(frames)} frames ({N_STEPS} steps, every {SAVE_EVERY})")
print(f"  u shape: {frames[0]['u'].shape}, v shape: {frames[0]['v'].shape}")
print(f"  pressure shape: {frames[0]['pressure'].shape}")

In [ ]:
# ── Module 1 — Animate pressure field. Export to MP4 ─────────────────────

# ── Single-panel figure ──
fig, ax = plt.subplots(1, 1, figsize=(10, 5), facecolor="#111111")
fig.suptitle(
    "Magnus Effect — Rotating Cylinder in Crossflow", color="white", fontsize=14, y=0.98
)
ax.set_facecolor("#111111")

p0 = frames[0]["pressure"]
im = ax.imshow(p0.T, origin="lower", cmap="magma", aspect="auto", extent=[0, 8, 0, 4])
ax.set_title("Pressure Field", color="white")
ax.set_xlabel("x")
ax.set_ylabel("y")
circle = plt.Circle(CENTER, RADIUS, color="white", fill=False, lw=1.5)
ax.add_patch(circle)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pressure", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")
fig.tight_layout(rect=[0, 0, 1, 0.93])


def update(idx):
    data = frames[idx]
    im.set_data(data["pressure"].T)
    im.set_clim(data["pressure"].min(), data["pressure"].max())
    return (im,)


anim = FuncAnimation(fig, update, frames=len(frames), interval=100, blit=True)

path = ASSETS / "magnus_effect.mp4"
anim.save(str(path), writer="ffmpeg", fps=10, dpi=150)
print(f"Saved {path}")

HTML(anim.to_jshtml())

In [ ]:
# ── Module 1 — Animate velocity streamlines. Export to MP4 ───────────────

x_s = np.linspace(0, 8, 256)
y_s = np.linspace(0, 4, 128)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
strm = ax.streamplot(
    x_s,
    y_s,
    frames[0]["u"].T,
    frames[0]["v"].T,
    color=frames[0]["velocity"].T,
    cmap="inferno",
    linewidth=0.8,
    density=1.5,
    arrowstyle="->",
    arrowsize=0.6,
)
ax.add_patch(Circle((2.0, 2.0), 0.3, color="white", fill=False, lw=2.0))
ax.set_xlim(0.5, 7.5)
ax.set_ylim(0.5, 3.5)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title(f"Velocity Streamlines — step {frames[0]['step']}")
cbar = plt.colorbar(strm.lines, ax=ax, label="Speed")
plt.tight_layout()
plt.show()


def animate_stream(i):
    ax.clear()
    strm = ax.streamplot(
        x_s,
        y_s,
        frames[i]["u"].T,
        frames[i]["v"].T,
        color=frames[i]["velocity"].T,
        cmap="inferno",
        linewidth=0.8,
        density=1.5,
        arrowstyle="->",
        arrowsize=0.6,
    )
    ax.add_patch(Circle((2.0, 2.0), 0.3, color="white", fill=False, lw=2.0))
    ax.set_xlim(0.5, 7.5)
    ax.set_ylim(0.5, 3.5)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(f"Velocity Streamlines — step {frames[i]['step']}")
    return (strm.lines,)


anim3 = FuncAnimation(fig, animate_stream, frames=len(frames), interval=80, blit=False)
anim3.save("assets/magnus_streamlines.mp4", writer="ffmpeg", dpi=120)
print("Saved assets/magnus_streamlines.mp4")
plt.close(fig)

# Module 2: The Peloton & Running Pack

## Wake Interaction & Drafting Efficiency

Drag accounts for up to 80% of the resistive force on a runner at sprint speeds.
By positioning behind another athlete, a runner enters their **wake** — a region
of reduced dynamic pressure — lowering the drag they must overcome.

We study three configurations with **rectangular runner cross-sections**
(0.3 × 0.5 units, rotated torso profile):

| Case | Setup | Expected Effect |
|------|-------|----------------|
| **A** — Solo | Single runner (baseline) | Reference drag |
| **B** — Inline Drafting | Two runners on same line, 1.0 unit apart | Trailing runner sees reduced drag |
| **C** — Echelon Offset | Two runners offset diagonally, 0.6 × 0.3 gap | Partial wake coverage |

### Setup

- **Grid**: 128 × 256, 4 × 8 unit domain, uniform inflow 1.0 m/s
- **Runner shape**: `geom.Box` — 0.3 wide × 0.5 tall
- **Solver**: ΦFlow incompressible Navier-Stokes, 80 steps × Δt = 0.3

In [ ]:
"""Module 2 — Three wake cases with rectangular runners and face-based drag."""

# ── Domain ───────────────────────────────────────────────────────────────────
BOX = geom.Box(x=8.0, y=4.0)
BOUNDARY = {"x-": vec(x=1.0, y=0.0), "x+": ZERO_GRADIENT, "y": 0}

# ── Runner shape ─────────────────────────────────────────────────────────────
HW, HH = 0.15, 0.25  # half-width (x), half-height (y) → 0.3 × 0.5


def runner_at(cx, cy):
    return geom.Box(x=(cx - HW, cx + HW), y=(cy - HH, cy + HH))


# ── Positions (8×4 domain) ──────────────────────────────────────────────────
case_a = [Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0))]

case_b = [
    Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0)),
    Obstacle(runner_at(3.5, 2.0), velocity=vec(x=0.0, y=0.0)),
]

case_c = [
    Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0)),
    Obstacle(runner_at(3.5, 2.35), velocity=vec(x=0.0, y=0.0)),
]

# ── Face-based drag masks ────────────────────────────────────────────────────
x_g = np.linspace(0, 8.0, 256)
y_g = np.linspace(0, 4.0, 128)
XX, YY = np.meshgrid(x_g, y_g, indexing="ij")


def face_drag_mask(cx, cy):
    up = (XX > cx - HW * 1.5) & (XX < cx - HW * 0.8) & (YY > cy - HH) & (YY < cy + HH)
    dn = (XX > cx + HW * 0.8) & (XX < cx + HW * 1.5) & (YY > cy - HH) & (YY < cy + HH)
    return up, dn


def box_drag(p_np, cx, cy):
    up, dn = face_drag_mask(cx, cy)
    return float(p_np[up].mean() - p_np[dn].mean()) if up.sum() and dn.sum() else 0.0


# ── Centered grid for velocity resampling ────────────────────────────────────
_centered = CenteredGrid(0, x=256, y=128, bounds=BOX)

# ── Simulation runner ────────────────────────────────────────────────────────
DT = 0.3
N_STEPS = 160
SAVE_EVERY = 2


def run_case(obstacles, label, drag_pos=None):
    v0 = StaggeredGrid((1.0, 0.0), BOUNDARY, x=256, y=128, bounds=BOX)
    v, _ = fluid.make_incompressible(v0, ())
    frames = []
    drag_series = []
    v = v0

    for i in trange(N_STEPS, desc=label):
        v = advect.semi_lagrangian(v, v, dt=DT)
        v, p = fluid.make_incompressible(v, obstacles)

        if i % SAVE_EVERY == 0:
            vel_mag = field.vec_length(v)
            vel_np = np.array(vel_mag.values.native("x", "y"))
            vc = v @ _centered
            u_c = np.array(vc.values.vector[0].native("x", "y"))
            v_c = np.array(vc.values.vector[1].native("x", "y"))
            frames.append(
                {
                    "velocity": vel_np,
                    "u": u_c,
                    "v": v_c,
                    "step": i,
                }
            )
            if drag_pos:
                drag_series.append(
                    box_drag(np.array(p.values.native("x", "y")), *drag_pos)
                )

    avg_drag = np.mean(drag_series[-len(drag_series) // 5 :]) if drag_series else 0.0
    return frames, avg_drag, drag_series


# ── Execute ──────────────────────────────────────────────────────────────────
print("Case A — Solo (baseline)")
frames_a, drag_a, _ = run_case(case_a, "Case A (solo)")

print("\nCase B — Inline drafting")
frames_b, drag_b, drag_b_s = run_case(case_b, "Case B (inline)", (3.5, 2.0))

print("\nCase C — Echelon offset")
frames_c, drag_c, drag_c_s = run_case(case_c, "Case C (echelon)", (3.5, 2.35))

print(f"\nDrag: A={drag_a:.4f}  B={drag_b:.4f}  C={drag_c:.4f}")
print(f"  B/A = {drag_b / drag_a:.3f}  C/A = {drag_c / drag_a:.3f}")

# ── Print comparison ──
print("\n" + "=" * 50)
print("Drag Comparison (relative to solo)")
print("=" * 50)
print(f"  A — Solo:                      1.000")
print(f"  B — Inline drafting:           {drag_b / drag_a:.3f}")
print(f"  C — Echelon offset:            {drag_c / drag_a:.3f}")
print(f"  Drag reduction (inline):       {(1 - drag_b / drag_a) * 100:.1f}%")
print(f"  Drag reduction (echelon):      {(1 - drag_c / drag_a) * 100:.1f}%")
print("=" * 50)

In [ ]:
"""Module 2 — Three-panel wake comparison animation + MP4 export."""

from matplotlib.patches import Rectangle

# ── Normalise frames ──
n_frames = min(len(frames_a), len(frames_b), len(frames_c))
trim = lambda f: f[:n_frames]
frames_a, frames_b, frames_c = trim(frames_a), trim(frames_b), trim(frames_c)

# ── 3-panel figure ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor="#111111")
fig.suptitle("Wake Comparison — Velocity Magnitude", color="white", fontsize=15, y=0.98)

labels = [
    f"A — Solo Runner (drag = 1.000)",
    f"B — Inline Drafting (drag = {drag_b / drag_a:.3f})",
    f"C — Echelon Offset (drag = {drag_c / drag_a:.3f})",
]

imgs = []
for ax, label, frames in zip(axes, labels, [frames_a, frames_b, frames_c]):
    ax.set_facecolor("#111111")
    ax.set_title(label, color="white", fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 8)
    # Mark runner positions
    for cx, cy in (
        [(2.0, 4.5)]
        if "Solo" in label
        else (
            [(2.0, 4.5), (2.0, 3.1)] if "Inline" in label else [(1.4, 4.5), (2.6, 3.1)]
        )
    ):
        ax.add_patch(plt.Circle((cx, cy), 0.2, color="white", fill=False, lw=1.2))
    im = ax.imshow(
        frames[0]["velocity"].T,
        origin="lower",
        cmap="inferno",
        aspect="auto",
        extent=[0, 4, 0, 8],
        vmin=0,
        vmax=1.5,
    )
    imgs.append(im)

fig.tight_layout(rect=[0, 0, 1, 0.93])

cbar = fig.colorbar(imgs[0], ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Velocity magnitude", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


def update(idx):
    for im, frames in zip(imgs, [frames_a, frames_b, frames_c]):
        im.set_data(frames[idx]["velocity"].T)
    fig.suptitle(
        f"Wake Comparison — Step {frames_a[idx]['step']}  "
        f"(t = {frames_a[idx]['step'] * DT:.1f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )
    return imgs


anim = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=True)

path = ASSETS / "wake_comparison.mp4"
anim.save(str(path), writer="ffmpeg", fps=12, dpi=150)
print(f"Exported: {path}")

# ── Drag summary table ──
fig2, ax_tbl = plt.subplots(figsize=(6, 2.5), facecolor="#111111")
ax_tbl.axis("off")
cols = ["Case", "Setup", "Drag (rel.)", "Reduction"]
rows = [
    ["A", "Solo runner", "1.000", "—"],
    [
        "B",
        "Inline drafting",
        f"{drag_b / drag_a:.3f}",
        f"{(1 - drag_b / drag_a) * 100:.0f}%",
    ],
    [
        "C",
        "Echelon offset",
        f"{drag_c / drag_a:.3f}",
        f"{(1 - drag_c / drag_a) * 100:.0f}%",
    ],
]
tbl = ax_tbl.table(cellText=rows, colLabels=cols, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.6)
for key, cell in tbl.get_celld().items():
    cell.set_facecolor("#1a1a1a")
    cell.set_text_props(color="white")
    if key[0] == 0:
        cell.set_facecolor("#333333")
        cell.set_text_props(color="white", weight="bold")
fig2.savefig(
    ASSETS / "drag_comparison.png", dpi=150, bbox_inches="tight", facecolor="#111111"
)
print(f"Saved: {ASSETS / 'drag_comparison.png'}")

from IPython.display import HTML

HTML(anim.to_jshtml())

In [ ]:
"""Module 2 — Three-panel streamlines comparison animation + MP4 export."""

from matplotlib.patches import Circle


# Re-run simulation to capture velocity components
def run_streamlines_case(obstacles, label):
    v0 = StaggeredGrid((1.0, 0.0), BOUNDARY, x=128, y=256, bounds=BOX)
    v, _ = fluid.make_incompressible(v0, ())
    frames = []
    v = v0
    for i in trange(N_STEPS, desc=label):
        v = advect.semi_lagrangian(v, v, dt=DT)
        v, p = fluid.make_incompressible(v, obstacles)
        if i % SAVE_EVERY == 0:
            vc = v @ CenteredGrid(0, x=128, y=256, bounds=BOX)
            u_c = np.array(vc.vector[0].values.native("x", "y"))
            v_c = np.array(vc.vector[1].values.native("x", "y"))
            vel_mag = field.vec_length(v)
            frames.append(
                {
                    "u": u_c,
                    "v": v_c,
                    "velocity": np.array(vel_mag.values.native("x", "y")),
                    "step": i,
                }
            )
    return frames


print("Re-running with velocity components for streamlines...")
sf_a = run_streamlines_case(case_a, "Case A")
sf_b = run_streamlines_case(case_b, "Case B")
sf_c = run_streamlines_case(case_c, "Case C")

n_frames = min(len(sf_a), len(sf_b), len(sf_c))
trim_f = lambda f: f[:n_frames]
sf_a, sf_b, sf_c = trim_f(sf_a), trim_f(sf_b), trim_f(sf_c)

x_s = np.linspace(0, 4, 128)
y_s = np.linspace(0, 8, 256)

fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor="#111111")
fig.suptitle("Wake Comparison — Streamlines", color="white", fontsize=15, y=0.98)

s_labels = [f"A — Solo Runner", f"B — Inline Drafting", f"C — Echelon Offset"]

strm_plots = []
for ax, label, frames, runners in zip(
    axes,
    s_labels,
    [sf_a, sf_b, sf_c],
    [[(2.0, 4.5)], [(2.0, 4.5), (2.0, 3.1)], [(1.4, 4.5), (2.6, 3.1)]],
):
    ax.set_facecolor("#111111")
    ax.set_title(label, color="white", fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 8)
    for cx, cy in runners:
        ax.add_patch(Circle((cx, cy), 0.2, color="white", fill=False, lw=1.2))
    s = ax.streamplot(
        x_s,
        y_s,
        frames[0]["u"].T,
        frames[0]["v"].T,
        color=frames[0]["velocity"].T,
        cmap="inferno",
        linewidth=0.8,
        density=1.2,
    )
    strm_plots.append(s)

fig.tight_layout(rect=[0, 0, 1, 0.93])


def update_s(idx):
    for ax, s, frames in zip(axes, strm_plots, [sf_a, sf_b, sf_c]):
        ax.clear()
        ax.set_facecolor("#111111")
        ax.set_xlim(0, 4)
        ax.set_ylim(0, 8)
        for cx, cy in (
            [(2.0, 4.5)],
            [(2.0, 4.5), (2.0, 3.1)],
            [(1.4, 4.5), (2.6, 3.1)],
        )[strm_plots.index(s)]:
            pass  # runners re-added per axis
        ax.streamplot(
            x_s,
            y_s,
            frames[idx]["u"].T,
            frames[idx]["v"].T,
            color=frames[idx]["velocity"].T,
            cmap="inferno",
            linewidth=0.8,
            density=1.2,
        )
    return tuple(strm_plots)


anim = FuncAnimation(fig, update_s, frames=n_frames, interval=80, blit=False)
anim.save("assets/wake_streamlines.mp4", writer="ffmpeg", dpi=120)
print("Saved assets/wake_streamlines.mp4")

HTML(anim.to_jshtml())